# Supplementary Code S2: reproduction notebook (Python)

Article: *Older adults with care needs and no helper in South Korea and China, 2013–2020: how measurement shapes the comparison*. This notebook recomputes the main results from the original survey files of KLoSA (Korean Longitudinal Study of Ageing) and CHARLS (China Health and Retirement Longitudinal Study) and compares them with the published values.

| Output | Article location |
|---|---|
| Respondents with need, number without a helper, weighted % with no helper (design-based 95% CI) | Table 3, Figure 2, Supplementary Tables S1 and S8b |
| Characteristics of respondents with need | Table 2 |
| Crude and age–sex standardized Korea-minus-China gaps among respondents with need (1,000 bootstrap replicates) | Table 4, Figure 3a (Korea 2014 / China 2013: Table 5, Supplementary Table S5) |
| Gaps under definition B, the 10 common activities, the matched five ADL items, age 70+, by sex | Figure 3a, Supplementary Tables S5 and S15, Text S13 |
| A-minus-B and A-minus-common differences, pairing effect, between-wave changes | Supplementary Table S7, Text S12 |
| Need prevalence and the share with need and no helper among all adults aged 65+, and standardized Korea-minus-China differences | Table 5, Figure 3b, Supplementary Table S15 |
| Post hoc CHARLS analyses: help counted only on activities with need, help questions answered, respondents with difficulty only, telephone help, institutional residents, age at interview, need on 2+ or 3+ of the 10 items among all adults | Supplementary Text S12 |

**Data.** The survey data cannot be redistributed. Register and download them yourself:
* CHARLS: https://charls.pku.edu.cn — the 2013, 2015, 2018 and 2020 wave archives and Harmonized CHARLS version D.
* KLoSA: https://survey.keis.or.kr — the Stata files of waves 5 (2014), 7 (2018) and 8 (2020); waves 6, 9 and 10 are optional context waves.

Put the files in one folder (in Colab, a Google Drive folder; subfolders allowed) and set `DATA_DIR` (input) and `OUT_DIR` (output) in Section 1. Keep **one** copy of each file: either the official archives or the extracted files. The notebook recognizes:

| Input | Accepted names |
|---|---|
| Harmonized CHARLS D | `H_CHARLS_D_Data.dta` (or inside `H_CHARLS_D_Data.zip`) |
| CHARLS 2013, 2015, 2018, 2020 | `Health_Status_and_Functioning.dta` and `Weights.dta` (2020 also `Demographic_Background.dta`) inside `CHARLS2013_Dataset.zip`, `CHARLS2015r.zip`, `CHARLS2018r.zip`, `CHARLS2020r.zip`, or in folders whose name contains the year, or with a year prefix such as `2018_Weights.dta` (unprefixed files outside a year-named folder are taken as 2015, as in Code S1) |
| KLoSA 2014, 2018, 2020 (required unless `CHARLS_ONLY`); 2016, 2022, 2024 (optional) | files whose names start with `str05`, `str07`, `str08` (`str06`, `str09`, `str10`) |

**CHARLS only.** Set `CHARLS_ONLY = True` in Section 1 to run without the KLoSA files. The notebook then checks the 109 published values that depend on CHARLS alone: 58 first computed with the original analysis scripts (Chinese counts, weighted percentages, standard errors, Table 2, need and no-helper shares among all adults and between-wave changes) and 51 post hoc values of Supplementary Text S12 that were computed with this notebook, so for them the comparison checks transcription only; Korean values, gaps, Chinese standardized percentages and the pairing effect need both surveys.

Keep the data in your own storage, do not send data or outputs to any external service, and never print or export individual-level rows; all outputs below are aggregate. CHARLS 2011 (a context point in Figure 2, with a different help question) is not reproduced here, and values for the optional Korean context waves are computed but not checked against the article.

**Agreement with the published values.** Point estimates should match at the published precision; with both surveys the comparison table at the end checks 219 published values (counts, percentages, standard errors, Table 2, standardized proportions and gaps, sensitivity estimates and the post hoc CHARLS analyses of Supplementary Text S12). Bootstrap intervals use a different random-number stream from the original Python analysis and can differ by a few tenths of a percentage point. The coding rules for definitions A and B are identical to Supplementary Code S1 (R). Run on the CHARLS files, the notebook reproduced all 58 independently computed CHARLS-only values and matched the 51 post hoc values it computes itself; the Korean values are checked when it is run on the KLoSA files.

**Software.** Written for Google Colab; also runs locally with Jupyter. Tested with Python 3.13.6 (pandas 2.3.3, NumPy 2.2.0, matplotlib 3.10.8). The files `codeS2_core.py` and `codeS2_analysis.py` in the same archive are copies of the code in cells 4 and 5, for reading or local use; the notebook does not import them.

## 1. Settings

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Folder that holds the CHARLS and KLoSA files (zip archives or .dta files, subfolders allowed)
DATA_DIR = os.environ.get('S2_DATA_DIR', '/content/drive/MyDrive/KLoSA_CHARLS_data' if IN_COLAB else os.path.abspath('KLoSA_CHARLS_data'))
# Folder for the output tables and figures
OUT_DIR = os.environ.get('S2_OUT_DIR', '/content/drive/MyDrive/CodeS2_outputs' if IN_COLAB else os.path.abspath('CodeS2_outputs'))
CHARLS_ONLY = os.environ.get('S2_CHARLS_ONLY', '0') == '1'   # True: run without the KLoSA files (Chinese values only)
B = int(os.environ.get('S2_BOOT', 1000))      # bootstrap replicates (article: 1,000)
SEED = 20261027
os.makedirs(OUT_DIR, exist_ok=True)
print('DATA_DIR =', repr(DATA_DIR), '| exists:', os.path.isdir(DATA_DIR))
print('OUT_DIR  =', repr(OUT_DIR), '| B =', B, '| CHARLS_ONLY =', CHARLS_ONLY)

## 2. Coding rules

* Age 65+ at each wave (KLoSA: the age variable `A002_age`, which the codebooks define as survey year minus birth year; CHARLS: survey year minus birth year, with birth year and sex from Harmonized CHARLS D, in 2020 supplemented from the 2020 demographic file when missing there).
* Valid activity items: at least 16 of 17 (KLoSA) or 10 of 11 (CHARLS, continence included). CHARLS 2013–2018 skip the six ADL questions when every answered mobility item db001–db009 shows no difficulty (db002 and db003 are asked only after difficulty on the previous item); these respondents are coded as having no ADL difficulty.
* Need: KLoSA codes 3 or 5 (needs partial or full help) on C201–C217; CHARLS rating 3 or higher on the ten items that have a help question (continence excluded). A = any ADL or IADL item; B = ADL items only (KLoSA C201–C207; CHARLS five items); common = the ten KLoSA items that match the CHARLS activities; matched ADL = dressing, bathing, eating, getting in or out of bed, toilet in both surveys; D (China only) = any item rated 2 or higher.
* No helper: KLoSA C218_01 = 56 ("no one helps"); CHARLS: no "yes" to any of the 10 item-level help questions (telephone help not counted). Respondents with need who answered no help question are excluded.
* Weights: KLoSA `wgt_ac` (`wgt_a` in wave 5); CHARLS `INDV_weight_ad2`. Respondents with a missing or zero weight are excluded.
* Standardization: 10 strata (65–69, 70–74, 75–79, 80–84, 85+ by sex); the standard is the mean of the two countries' weighted stratum shares among respondents with need under the definition analysed.
* Bootstrap: 1,000 replicates; communities resampled in China, respondents in Korea, as in the published analysis (the enumeration-district identifier ed is used only in the KLoSA checks of Section 9). All statistics of one replicate use the same resamples, so differences between definitions and pairings are paired.

In [ ]:
# Supplementary Code S2, core functions: data loading and coding rules (KLoSA, South Korea; CHARLS, China).
# Article: Older adults with care needs and no helper in South Korea and China, 2013-2020: how measurement shapes the comparison
# The coding rules follow the manuscript Methods, Supplementary Table S3 and Supplementary Code S1 (R).
import io, os, re, zipfile
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------- file discovery
def _members(data_dir):
    """Yield (display_name, opener) for every .dta file in data_dir, including files inside .zip archives."""
    for root, dirs, files in os.walk(data_dir):
        dirs[:] = sorted(d for d in dirs if not os.path.ismount(os.path.join(root, d)))   # do not walk into a mounted Drive
        for f in sorted(files):
            p = os.path.join(root, f)
            if f.lower().endswith('.dta'):
                yield os.path.relpath(p, data_dir), (lambda p=p: open(p, 'rb'))
            elif f.lower().endswith('.zip'):
                try:
                    z = zipfile.ZipFile(p)
                except zipfile.BadZipFile:
                    continue
                for m in z.namelist():
                    if m.lower().endswith('.dta') and not m.startswith('__MACOSX'):
                        yield f'{os.path.relpath(p, data_dir)}::{m}', (lambda z=z, m=m: io.BytesIO(z.read(m)))

def index_files(data_dir, require_klosa=True):
    """Map each needed input to one file. Raises an error that lists candidates when a match is ambiguous."""
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f'DATA_DIR does not exist: {data_dir!r} (check spaces and folder names)')
    files = list(_members(data_dir))
    if not files:
        raise FileNotFoundError(f'No .dta or .zip files found under {data_dir!r}')
    def pick(test, label, required=True):
        hits = [(n, o) for n, o in files if test(n)]
        if len(hits) > 1:   # the same file both inside a zip and extracted next to it: use the extracted copy
            plain = [h for h in hits if '::' not in h[0]]
            if len(plain) == 1:
                hits = plain
        if len(hits) == 1:
            return hits[0]
        if not hits and not required:
            return None
        raise FileNotFoundError(f'{label}: expected one file, found {len(hits)}: {[n for n, _ in hits]}. '
                                'Keep one copy of each file under DATA_DIR (either the zip or the extracted files).')
    parts = lambda n: re.split(r'[/\\]|::', n)
    base = lambda n: parts(n)[-1].lower()
    container = lambda n: parts(n)[-2].lower() if len(parts(n)) > 1 else ''   # zip name or immediate folder
    def charls(year, stem):
        # wave = year in the zip or folder that holds the file (CHARLS2015r.zip, 2015/) or a year prefix (2015_Weights.dta);
        # an unprefixed file outside any year-named container is the 2015 file (naming used by Code S1)
        def t(n):
            b, c = base(n), container(n)
            if not re.fullmatch(rf'(\d{{4}}_)?{stem.lower()}\.dta', b):
                return False
            if b.startswith(f'{year}_'):
                return True
            if re.match(r'\d{4}_', b):
                return False
            if str(year) in c:
                return True
            return year == 2015 and not re.search(r'20(11|13|15|18|20)', c)
        return t
    out = {'H': pick(lambda n: base(n) == 'h_charls_d_data.dta', 'Harmonized CHARLS D')}
    for y in (2013, 2015, 2018, 2020):
        out[f'CN{y}_hsf'] = pick(charls(y, 'Health_Status_and_Functioning'), f'CHARLS {y} health status')
        out[f'CN{y}_wt'] = pick(charls(y, 'Weights'), f'CHARLS {y} weights')
    out['CN2020_dem'] = pick(charls(2020, 'Demographic_Background'), 'CHARLS 2020 demographic background')
    for y, w in KLOSA_WAVES.items():
        out[f'KR{y}'] = pick(lambda n, w=w: re.match(rf'str{w}', base(n)) is not None, f'KLoSA {y} (str{w}*.dta)',
                             required=require_klosa and y in (2014, 2018, 2020))
    return out

def read_dta(entry, columns=None):
    name, opener = entry
    with opener() as fh:
        df = pd.read_stata(fh, convert_categoricals=False, columns=columns)
    return df

# ----------------------------------------------------------------------------- CHARLS
CHARLS_YEARS = (2013, 2015, 2018, 2020)

def fid(s):
    """Harmonize CHARLS IDs: 11-digit IDs of 2013-2015 get a 0 after the 9-digit household part (as in Code S1)."""
    s = s.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    return np.where(s.str.len() == 11, s.str[:9] + '0' + s.str[9:], s)

def charls_items(year):
    if year == 2020:
        adl6 = ['db001', 'db003', 'db005', 'db007', 'db009', 'db011']   # db011 = continence (no help question)
        iadl = ['db012', 'db014', 'db016', 'db022', 'db020']            # chores, hot meals, groceries, money, medications
        helpv = [f'db{int(v[2:]) + 1:03d}' for v in adl6[:5] + iadl]
    else:
        adl6 = [f'db0{i}' for i in range(10, 16)]                         # db015 = continence
        iadl = [f'db0{i}' for i in range(16, 21)]
        helpv = [v + '_w2' for v in adl6[:5] + iadl]
    return adl6, iadl, helpv

def load_charls(files, H, year):
    adl6, iadl, helpv = charls_items(year)
    mobv = [f'db00{i}' for i in range(1, 10)] if year != 2020 else []
    phonev = 'db019' if year == 2020 else 'db035_w2'   # help with telephone calls (not part of the main help rule)
    d = read_dta(files[f'CN{year}_hsf'])
    missing = [c for c in ['ID'] + adl6 + iadl + helpv if c not in d.columns]
    if missing:
        raise KeyError(f'CHARLS {year} health file lacks {missing}')
    d['ID'] = fid(d['ID'])
    d = d.drop_duplicates('ID')
    num = lambda c: pd.to_numeric(d[c], errors='coerce') if c in d.columns else pd.Series(np.nan, index=d.index)
    x = pd.DataFrame({c: num(c).where(num(c).between(1, 4)) for c in adl6 + iadl})
    skip = pd.Series(False, index=d.index)
    if year != 2020:   # ADL block skipped by design: every answered mobility item db001-db009 is 1 (db002 and db003
                       # are asked only after difficulty on the previous item, so they are missing by design)
        mob = pd.DataFrame({c: num(c) for c in mobv if c in d.columns})
        all_one = (mob.isna() | (mob == 1)).all(axis=1)
        any_mob = mob.notna().any(axis=1)
        skip = (x[adl6].notna().sum(axis=1) == 0) & all_one & any_mob & (x[iadl].notna().sum(axis=1) >= len(iadl) - 1)
        x.loc[skip, adl6] = 1
    items10 = adl6[:5] + iadl
    valid = x.notna().sum(axis=1) >= len(adl6 + iadl) - 1          # at least 10 of 11 (continence included)
    h = pd.DataFrame({c: num(c) for c in helpv})
    out = pd.DataFrame({
        'ID': d['ID'].values, 'valid': valid.values, 'skipped': skip.values,
        'needA': (x[items10] >= 3).any(axis=1).values,
        'needB': (x[adl6[:5]] >= 3).any(axis=1).values,
        'needD': (x[items10] >= 2).any(axis=1).values,
        'n_lim': (x[items10] >= 3).sum(axis=1).values,
        'helped': (h == 1).any(axis=1).values,
        'hans': h.isin([1, 2]).any(axis=1).values,
        # post hoc: help counted only on activities with need (rating 3 or higher); number of help questions answered
        'helped_need': ((h.to_numpy() == 1) & (x[items10] >= 3).to_numpy()).any(axis=1),
        'hans_need': (h.isin([1, 2]).to_numpy() & (x[items10] >= 3).to_numpy()).any(axis=1),
        'nq': h.isin([1, 2]).sum(axis=1).values,
        'helped_phone': (num(phonev) == 1).values,
    })
    out['needC'] = out['needA']      # common activities: all 10 CHARLS items have a KLoSA counterpart
    out['needM'] = out['needB']      # matched five ADL items: same five items in CHARLS
    # severity on scales shared by both surveys: number of the 10 common activities, and of the
    # five matched ADL items, with need
    out['n_common'] = (x[items10] >= 3).sum(axis=1).values
    out['n_adl5'] = (x[adl6[:5]] >= 3).sum(axis=1).values
    out['needC3'] = out['n_common'] >= 3
    out['needM2'] = out['n_adl5'] >= 2
    W = read_dta(files[f'CN{year}_wt'], columns=['ID', 'INDV_weight_ad2', 'communityID'])
    W['ID'] = fid(W['ID'])
    W = W.drop_duplicates('ID').rename(columns={'INDV_weight_ad2': 'w', 'communityID': 'cl'})
    W['w'] = pd.to_numeric(W['w'], errors='coerce')
    W['cl'] = W['cl'].astype(str).str.strip()
    out = out.merge(W, on='ID', how='left').merge(H, on='ID', how='left')
    if year == 2020:
        D = read_dta(files['CN2020_dem'], columns=['ID', 'ba003_1', 'ba001', 'ba007'])
        D['ID'] = fid(D['ID'])
        D = D.drop_duplicates('ID')
        out = out.merge(D, on='ID', how='left')
        out['rabyear'] = out['rabyear'].fillna(pd.to_numeric(out['ba003_1'], errors='coerce'))
        out['ragender'] = out['ragender'].fillna(pd.to_numeric(out['ba001'], errors='coerce'))
        out['inst'] = pd.to_numeric(out['ba007'], errors='coerce').isin([2, 3])   # nursing home or hospital
        out['agey'] = np.nan
    else:   # Harmonized CHARLS: lives in an institution at interview; age at interview (wave 2 = 2013, 3 = 2015, 4 = 2018)
        wv = {2013: 2, 2015: 3, 2018: 4}[year]
        out['inst'] = out[f'r{wv}nhmliv'] == 1
        out['agey'] = out[f'r{wv}agey']
    out['age'] = year - out['rabyear']
    out['fem'] = (out['ragender'] == 2).astype(int)
    out['nh'] = (~out['helped']).astype(int)
    out['country'], out['year'] = 'CN', year
    base = out[out['valid'] & (out['age'] >= 65) & out['w'].notna() & (out['w'] > 0)].copy()
    return base

def load_harmonized(files):
    extra = [f'r{w}{v}' for w in (2, 3, 4) for v in ('nhmliv', 'agey')]
    H = read_dta(files['H'], columns=['ID', 'rabyear', 'ragender'] + extra)
    H['ID'] = fid(H['ID'])
    for c in ['rabyear', 'ragender'] + extra:
        H[c] = pd.to_numeric(H[c], errors='coerce')
    return H.drop_duplicates('ID')

# ----------------------------------------------------------------------------- KLoSA
KLOSA_WAVES = {2014: '05', 2016: '06', 2018: '07', 2020: '08', 2022: '09', 2024: '10'}
KR_COMMON = [1, 3, 4, 5, 6, 9, 10, 14, 15, 17]   # C2xx items matched to the 10 CHARLS activities (Table S3)
KR_MATCHED_ADL = [1, 3, 4, 5, 6]                 # dressing, bathing, eating, bed, toilet

def load_klosa(files, year):
    wv = 'w' + KLOSA_WAVES[year]
    d = read_dta(files[f'KR{year}'])
    cols = {c.lower(): c for c in d.columns}
    need_vars = [f'C2{i:02d}' for i in range(1, 18)] + ['C218_01', 'A002_age', 'gender1']
    missing = [v for v in need_vars if (wv + v).lower() not in cols]
    if not any(c in cols for c in ((wv + 'wgt_ac').lower(), (wv + 'wgt_a').lower())):
        missing.append('wgt_ac/wgt_a')
    if missing:
        raise KeyError(f'KLoSA {year} file lacks {[wv + v for v in missing]}')
    g = lambda n: pd.to_numeric(d[cols[(wv + n).lower()]], errors='coerce') if (wv + n).lower() in cols else pd.Series(np.nan, index=d.index)
    wname = (wv + 'wgt_ac') if (wv + 'wgt_ac').lower() in cols else (wv + 'wgt_a')
    C = pd.DataFrame({i: g(f'C2{i:02d}') for i in range(1, 18)})
    need = C.isin([3, 5])
    helper = g('C218_01')
    out = pd.DataFrame({
        'age': g('A002_age'), 'fem': (g('gender1') == 5).astype(int), 'w': g(wname[len(wv):]),
        'valid': C.notna().sum(axis=1) >= 16,
        'needA': need.any(axis=1), 'needB': need[list(range(1, 8))].any(axis=1),
        'needC': need[KR_COMMON].any(axis=1), 'needM': need[KR_MATCHED_ADL].any(axis=1),
        'needD': False, 'n_lim': need.sum(axis=1),
        'nh': (helper == 56).astype(int), 'hans': helper.notna(),
        'n_common': need[KR_COMMON].sum(axis=1), 'n_adl5': need[KR_MATCHED_ADL].sum(axis=1),
        # for klosa_diagnostics() only: raw helper code, 2014 refresh-panel flag, LTCI use (C320n, not the hospital item
        # C320) and the enumeration district (ed) listed in the 2014-2022 codebooks; absent variables stay missing
        'helper': helper, 'oldnew': g('oldnew'), 'ltci': g('C320n'), 'ltci_type': g('C327n'), 'proxy_c2': g('C225'),
        'ed': d[cols['ed']] if 'ed' in cols else pd.Series(np.nan, index=d.index),
    })
    out['needC3'] = out['n_common'] >= 3
    out['needM2'] = out['n_adl5'] >= 2
    out['cl'] = np.arange(len(out)).astype(str)     # respondents independent, as in the published analysis (ed: klosa_diagnostics only)
    out['country'], out['year'] = 'KR', year
    return out[out['valid'] & (out['age'] >= 65) & out['w'].notna() & (out['w'] > 0)].copy()

# ----------------------------------------------------------------------------- estimation
AGE_CUTS = [64, 69, 74, 79, 84, 200]

def add_strata(df):
    band = pd.cut(df['age'], AGE_CUTS, labels=False)      # 0..4: 65-69 ... 85+
    df['st'] = (band * 2 + df['fem']).astype(int)          # 10 strata
    return df

def wmean_ci(df, var, cluster=True):
    """Weighted mean of a 0/1 variable with a linearized SE (clusters = 'cl'; one stratum, with replacement)."""
    w, y = df['w'].to_numpy(float), df[var].to_numpy(float)
    p = (w * y).sum() / w.sum()
    z = w * (y - p) / w.sum()
    u = pd.Series(z).groupby(df['cl'].to_numpy()).sum().to_numpy() if cluster else z
    n = len(u)
    se = np.sqrt(n / (n - 1) * (u ** 2).sum())
    return 100 * p, 100 * se, 100 * (p - 1.96 * se), 100 * (p + 1.96 * se)

def stratum_sums(st, w, nh, sel, mult=None):
    m = w * sel if mult is None else w * sel * mult
    num = np.bincount(st, weights=m * nh, minlength=10)
    den = np.bincount(st, weights=m, minlength=10)
    return num, den

def std_gap_from_sums(nk, dk, nc, dc):
    """Direct standardization: standard = mean of the two countries' weighted stratum shares (Code S1)."""
    share_k, share_c = dk / dk.sum(), dc / dc.sum()
    ok = (dk > 0) & (dc > 0)
    std = (share_k + share_c) / 2
    ww = std * ok / (std * ok).sum()
    with np.errstate(invalid='ignore', divide='ignore'):
        rk, rc = np.where(ok, nk / dk, 0), np.where(ok, nc / dc, 0)
    return 100 * (rk * ww).sum(), 100 * (rc * ww).sum(), 100 * ((rk - rc) * ww).sum()

class Wave:
    """Arrays for one survey wave, ready for fast bootstrap re-weighting."""
    def __init__(self, df, by_cluster):
        df = add_strata(df.reset_index(drop=True))
        self.df = df
        self.st = df['st'].to_numpy()
        self.w = df['w'].to_numpy(float)
        self.nh = df['nh'].to_numpy(float)
        self.hans = df['hans'].to_numpy(bool)
        codes, uniq = pd.factorize(df['cl'])
        self.cl, self.ncl = codes, len(uniq)
        self.by_cluster = by_cluster
    def sel(self, need, all_adults=False):
        nd = self.df[need].to_numpy(bool)
        if all_adults:     # all adults 65+: outcome = need and no helper; drop those with need who skipped the help question
            return ~(nd & ~self.hans), (nd & self.hans).astype(float) * self.nh
        return nd & self.hans, self.nh
    def draw(self, rng):
        if self.by_cluster:
            return np.bincount(rng.integers(0, self.ncl, self.ncl), minlength=self.ncl)[self.cl].astype(float)
        return np.bincount(rng.integers(0, len(self.w), len(self.w)), minlength=len(self.w)).astype(float)
    def sums(self, need, mult=None, all_adults=False, extra=None):
        sel, y = self.sel(need, all_adults)
        if extra is not None:
            sel = sel & extra
        return stratum_sums(self.st, self.w, y, sel.astype(float), mult)

def gap(kr, cn, need, mk=None, mc=None, all_adults=False, ek=None, ec=None):
    nk, dk = kr.sums(need, mk, all_adults, ek)
    nc, dc = cn.sums(need, mc, all_adults, ec)
    return std_gap_from_sums(nk, dk, nc, dc)

def crude_gap(kr, cn, need, mk=None, mc=None):
    nk, dk = kr.sums(need, mk)
    nc, dc = cn.sums(need, mc)
    return 100 * (nk.sum() / dk.sum() - nc.sum() / dc.sum())

In [ ]:
# Supplementary Code S2, analysis functions: tables, bootstrap intervals and comparison with the published values.
# Article: Older adults with care needs and no helper in South Korea and China, 2013-2020: how measurement shapes the comparison
# Pairs or waves whose files are absent (for example, a run with the CHARLS files only) are skipped.
import numpy as np
import pandas as pd

PAIRS = {  # label: (Korean wave, Chinese wave, role)
    'KR2014/CN2015': (2014, 2015, 'main'),
    'KR2018/CN2018': (2018, 2018, 'main'),
    'KR2020/CN2020': (2020, 2020, 'sensitivity'),
    'KR2014/CN2013': (2014, 2013, 'sensitivity'),
}
DEF_LABEL = {'needA': 'A: any ADL/IADL need', 'needB': 'B: ADL need only', 'needC': 'Common 10 activities',
             'needM': 'Matched five ADL items', 'needD': 'D: any difficulty (China only)',
             'needC3': 'Need on 3+ of the 10 common activities', 'needM2': 'Need on 2+ of the 5 matched ADL items'}
GAP_DEFS = ['needA', 'needB', 'needC', 'needM', 'needC3', 'needM2']   # definitions compared between countries

def point_estimates(W, wmean_ci, gap, crude_gap):
    """W: dict of Wave objects keyed 'KR2014', 'CN2015', ... Returns dict of point estimates."""
    est = {}
    for key, wv in W.items():
        for need in GAP_DEFS + ['needD']:
            if key.startswith('KR') and need == 'needD':
                continue
            g = wv.df[wv.df[need] & wv.df['hans']]
            if len(g) == 0:
                continue
            p, se, lo, hi = wmean_ci(g, 'nh', cluster=key.startswith('CN'))
            est[('prop', key, need)] = dict(n=len(g), no_helper=int(g['nh'].sum()), pct=p, se=se, lo=lo, hi=hi,
                                            n70=int((g['age'] >= 70).sum()))
    for lab, (ky, cy, _) in PAIRS.items():
        if f'KR{ky}' not in W or f'CN{cy}' not in W:
            continue
        kr, cn = W[f'KR{ky}'], W[f'CN{cy}']
        for need in GAP_DEFS:
            k, c, g_ = gap(kr, cn, need)
            est[('gap', lab, need)] = dict(kr_std=k, cn_std=c, gap=g_, crude=crude_gap(kr, cn, need))
        for need in ['needB', 'needC', 'needM']:   # population denominator under the other definitions
            k, c, g_ = gap(kr, cn, need, all_adults=True)
            est[('all65', lab, need)] = dict(kr_std=k, cn_std=c, gap=g_)
        est[('gap70', lab, 'needA')] = dict(gap=gap(kr, cn, 'needA', ek=kr.df['age'].to_numpy() >= 70,
                                                     ec=cn.df['age'].to_numpy() >= 70)[2])
        for sex, f in (('men', 0), ('women', 1)):
            est[(f'gap_{sex}', lab, 'needA')] = dict(gap=gap(kr, cn, 'needA', ek=kr.df['fem'].to_numpy() == f,
                                                              ec=cn.df['fem'].to_numpy() == f)[2])
        k, c, g_ = gap(kr, cn, 'needA', all_adults=True)
        est[('all65', lab, 'needA')] = dict(kr_std=k, cn_std=c, gap=g_)
    return est

def characteristics(W):
    rows = []
    for key in ['KR2014', 'CN2015', 'KR2018', 'CN2018', 'KR2020', 'CN2020']:
        if key not in W:
            continue
        g = W[key].df
        g = g[g['needA'] & g['hans']]
        w = g['w']
        rows.append({'wave': key, 'n (unweighted)': len(g), 'Mean age, years': (w * g['age']).sum() / w.sum(),
                     'Female, %': 100 * (w * g['fem']).sum() / w.sum(),
                     'Aged 75 or older, %': 100 * (w * (g['age'] >= 75)).sum() / w.sum()})
    return pd.DataFrame(rows).set_index('wave').T

def all_adults_shares(W, needs=('needA',)):
    """Crude weighted need prevalence and share with need and no helper among all adults 65+ (standardized
    differences use the mean of the two countries' age-sex distributions among all adults 65+ as the standard)."""
    rows = []
    for key, wv in W.items():
        for need in needs:
            d = wv.df[~(wv.df[need] & ~wv.df['hans'])]
            w = d['w']
            lab = '' if need == 'needA' else f' [{need}]'
            rows.append({'wave': key, 'definition': need, 'n aged 65+': len(d),
                         'Need (definition A), weighted %' if need == 'needA' else 'Need, weighted %': 100 * (w * d[need]).sum() / w.sum(),
                         'Need and no helper, weighted %': 100 * (w * (d[need] & d['hans']) * d['nh']).sum() / w.sum()})
    out = pd.DataFrame(rows)
    return out.set_index('wave') if len(needs) == 1 else out

def klosa_diagnostics(W, wmean_ci):
    """Checks on the KLoSA files for the authors, outside the automated comparison with published values. Code 57 of C218_01 is listed
    in the 2018 and 2020 codebooks but not offered by the questionnaire (the main rule counts only 56 as no helper); the 2014
    refresh panel (oldnew = 2) should have nobody aged 65 or older; LTCI use (C320n = 1) among respondents with need was
    published as 4.3%, 7.9% and 14.0%, and C327n = 1 marks institutional LTCI benefits (institutional residents); C225 = 4 marks a
    proxy interview for the ADL and helper module (published without proxies: 26.0%, 29.0% and 36.4%); design effects compare SEs clustered by enumeration district (ed) with the published
    SEs, which treat respondents as independent. The unweighted share shows how much of a change between waves (for example
    2014 to 2016, when the weight variable changed from wgt_a to wgt_ac) comes from the weights."""
    rows = {}
    for key in sorted(k for k in W if k.startswith('KR')):
        d = W[key].df
        m = d[d['needA'] & d['hans']]
        r = {'n with need (A)': len(m),
             '% no helper (A), unweighted': 100 * float(m['nh'].mean()),
             'C218_01 = 57 among them': int((m['helper'] == 57).sum()),
             'refresh panel (oldnew = 2), aged 65+': int((d['oldnew'] == 2).sum()),
             'LTCI use (C320n = 1) among those with need, weighted %':
                 100 * float((m['w'] * (m['ltci'] == 1)).sum() / m['w'].sum()) if m['ltci'].notna().any() else np.nan,
             'institutional LTCI benefit (C327n = 1) among them, n': int((m['ltci_type'] == 1).sum()),
             'no helper among institutional LTCI users, n': int(((m['ltci_type'] == 1) & (m['nh'] == 1)).sum()),
             '% no helper (A), all': 100 * float((m['w'] * m['nh']).sum() / m['w'].sum()),
             '% no helper (A), institutional LTCI users excluded': 100 * float((m['w'] * m['nh'])[m['ltci_type'] != 1].sum() / m['w'][m['ltci_type'] != 1].sum()),
             'proxy interview for the ADL and helper module (C225 = 4), n': int((m['proxy_c2'] == 4).sum()),
             '% no helper (A), proxy interviews excluded': 100 * float((m['w'] * m['nh'])[m['proxy_c2'] != 4].sum() / m['w'][m['proxy_c2'] != 4].sum())}
        for need in ('needA', 'needB'):
            g = d[d[need] & d['hans']].copy()
            _, se_ind, _, _ = wmean_ci(g, 'nh', cluster=False)
            r[f'{need}: SE, respondents independent'] = se_ind
            if len(g) and g['ed'].notna().all():
                g['cl'] = g['ed'].astype(str)
                _, se_ed, _, _ = wmean_ci(g, 'nh', cluster=True)
                r[f'{need}: districts (ed)'] = g['ed'].nunique()
                r[f'{need}: SE, clustered by ed'] = se_ed
                r[f'{need}: design effect'] = (se_ed / se_ind) ** 2
            else:
                r[f'{need}: design effect'] = np.nan
        rows[key] = r
    return pd.DataFrame(rows)

BOOT_WAVES = ('CN2013', 'CN2015', 'CN2018', 'CN2020', 'KR2014', 'KR2018', 'KR2020')

def bootstrap(W, gap, crude_gap, B=1000, seed=20261027, progress=None):
    """Resample communities (China) and respondents (Korea) independently per wave; every statistic in one
    replicate uses the same resamples, so differences between definitions and pairings are paired."""
    rng = np.random.default_rng(seed)
    reps = []
    for b in range(B):
        M = {k: W[k].draw(rng) for k in BOOT_WAVES if k in W}   # fixed order: optional context waves never shift the stream
        r = {}
        for lab, (ky, cy, _) in PAIRS.items():
            kk, cc = f'KR{ky}', f'CN{cy}'
            if kk not in W or cc not in W:
                continue
            kr, cn, mk, mc = W[kk], W[cc], M[kk], M[cc]
            for need in GAP_DEFS:
                r[('gap', lab, need)] = gap(kr, cn, need, mk, mc)[2]
            for need in ['needB', 'needC', 'needM']:
                r[('all65', lab, need)] = gap(kr, cn, need, mk, mc, all_adults=True)[2]
            r[('crude', lab, 'needA')] = crude_gap(kr, cn, 'needA', mk, mc)
            r[('gap70', lab, 'needA')] = gap(kr, cn, 'needA', mk, mc, ek=kr.df['age'].to_numpy() >= 70,
                                             ec=cn.df['age'].to_numpy() >= 70)[2]
            for sex, f in (('men', 0), ('women', 1)):
                r[(f'gap_{sex}', lab, 'needA')] = gap(kr, cn, 'needA', mk, mc, ek=kr.df['fem'].to_numpy() == f,
                                                      ec=cn.df['fem'].to_numpy() == f)[2]
            r[('all65', lab, 'needA')] = gap(kr, cn, 'needA', mk, mc, all_adults=True)[2]
            r[('A-B', lab, '')] = r[('gap', lab, 'needA')] - r[('gap', lab, 'needB')]
            r[('A-common', lab, '')] = r[('gap', lab, 'needA')] - r[('gap', lab, 'needC')]
        for need in (['needA', 'needB'] if ('gap', 'KR2014/CN2013', 'needA') in r and ('gap', 'KR2014/CN2015', 'needA') in r else []):
            r[('pairing', 'CN2013 vs CN2015', need)] = r[('gap', 'KR2014/CN2013', need)] - r[('gap', 'KR2014/CN2015', need)]
        for country, waves in (('CN', [2013, 2015, 2018, 2020]), ('KR', [2014, 2018, 2020])):
            for a, b2 in zip(waves[:-1], waves[1:]):
                if f'{country}{a}' not in W or f'{country}{b2}' not in W:
                    continue
                for need in ['needA', 'needB']:
                    later, earlier = W[f'{country}{b2}'], W[f'{country}{a}']
                    r[('change', f'{country}{a}->{b2}', need)] = gap(later, earlier, need, M[f'{country}{b2}'], M[f'{country}{a}'])[2]
        reps.append(r)
        if progress and (b + 1) % 100 == 0:
            progress(b + 1)
    return pd.DataFrame(reps)

def summarize(est, reps, W, gap):
    """Point estimates with 2.5th-97.5th percentile bootstrap intervals."""
    ci = lambda key: (np.percentile(reps[key], 2.5), np.percentile(reps[key], 97.5))
    rows = []
    for key in reps.columns:
        kind, lab, need = key
        if kind in ('gap', 'gap70', 'gap_men', 'gap_women', 'all65'):
            pe = est[(kind, lab, need)]['gap']
        elif kind == 'crude':
            pe = est[('gap', lab, 'needA')]['crude']
        elif kind == 'A-B':
            pe = est[('gap', lab, 'needA')]['gap'] - est[('gap', lab, 'needB')]['gap']
        elif kind == 'A-common':
            pe = est[('gap', lab, 'needA')]['gap'] - est[('gap', lab, 'needC')]['gap']
        elif kind == 'pairing':
            pe = est[('gap', 'KR2014/CN2013', need)]['gap'] - est[('gap', 'KR2014/CN2015', need)]['gap']
        elif kind == 'change':
            c = lab[:2]; a, b2 = lab[2:].split('->')
            pe = gap(W[f'{c}{b2}'], W[f'{c}{a}'], need)[2]
        lo, hi = ci(key)
        rows.append({'statistic': kind, 'pair_or_change': lab, 'definition': need, 'estimate': pe, 'lo': lo, 'hi': hi})
    return pd.DataFrame(rows)

# Published values, written exactly as published (manuscript Tables 2-4; Supplementary Tables S1, S5, S7, S8b,
# S15; Texts S12-S13). A reproduced value matches when it rounds to the published value at the published precision.
PUB_POINTS = {
    # weighted % with no helper and its SE (S8b), counts (S8b, S13)
    **{('pct', w, 'needA'): v for w, v in [('KR2014', '24.24'), ('KR2018', '27.50'), ('KR2020', '34.33'), ('CN2013', '13.39'),
                                            ('CN2015', '16.78'), ('CN2018', '14.88'), ('CN2020', '11.23')]},
    **{('pct', w, 'needB'): v for w, v in [('KR2014', '16.34'), ('KR2018', '17.08'), ('KR2020', '18.77'), ('CN2013', '12.86'),
                                            ('CN2015', '13.17'), ('CN2018', '9.66'), ('CN2020', '5.74')]},
    **{('pct', w, 'needD'): v for w, v in [('CN2013', '31.8'), ('CN2015', '37.3'), ('CN2018', '33.2'), ('CN2020', '40.9')]},
    **{('pct', w, 'needC'): v for w, v in [('KR2014', '22.3'), ('KR2018', '24.1'), ('KR2020', '32.2')]},
    **{('se', w, 'needA'): v for w, v in [('KR2014', '1.71'), ('KR2018', '1.90'), ('KR2020', '2.09'), ('CN2013', '1.10'),
                                           ('CN2015', '1.36'), ('CN2018', '1.00'), ('CN2020', '0.94')]},
    **{('se', w, 'needB'): v for w, v in [('KR2014', '2.21'), ('KR2018', '2.54'), ('KR2020', '2.95')]},
    **{('n', w, 'needA'): v for w, v in [('KR2014', 749), ('KR2018', 739), ('KR2020', 718), ('CN2013', 1425), ('CN2015', 1776),
                                          ('CN2018', 2082), ('CN2020', 1869)]},
    **{('n', w, 'needB'): v for w, v in [('KR2014', 324), ('KR2018', 307), ('KR2020', 280), ('CN2013', 587), ('CN2015', 761),
                                          ('CN2018', 855), ('CN2020', 844)]},
    **{('n', w, 'needC'): v for w, v in [('KR2014', 684), ('KR2018', 662), ('KR2020', 634)]},
    **{('n70', w, 'needA'): v for w, v in [('KR2014', 661), ('KR2018', 675), ('KR2020', 659), ('CN2015', 1262), ('CN2018', 1455),
                                            ('CN2020', 1359)]},
    **{('nohelp', w, 'needA'): v for w, v in [('KR2014', 182), ('KR2018', 204), ('KR2020', 224), ('CN2013', 206), ('CN2015', 302),
                                               ('CN2018', 320), ('CN2020', 235)]},
    **{('nohelp', w, 'needB'): v for w, v in [('KR2014', 51), ('KR2018', 46), ('KR2020', 47), ('CN2013', 79), ('CN2015', 112),
                                               ('CN2018', 87), ('CN2020', 55)]},
    # standardized proportions (S8b)
    **{('kr_std', p, d): v for (p, d), v in [(('KR2014/CN2015', 'needA'), '24.37'), (('KR2018/CN2018', 'needA'), '27.49'),
        (('KR2020/CN2020', 'needA'), '35.08'), (('KR2014/CN2013', 'needA'), '24.44'), (('KR2014/CN2015', 'needB'), '17.20'),
        (('KR2018/CN2018', 'needB'), '18.21'), (('KR2020/CN2020', 'needB'), '19.95'), (('KR2014/CN2013', 'needB'), '17.19')]},
    **{('cn_std', p, d): v for (p, d), v in [(('KR2014/CN2015', 'needA'), '16.26'), (('KR2018/CN2018', 'needA'), '13.99'),
        (('KR2020/CN2020', 'needA'), '10.74'), (('KR2014/CN2013', 'needA'), '12.63'), (('KR2014/CN2015', 'needB'), '12.49'),
        (('KR2018/CN2018', 'needB'), '8.87'), (('KR2020/CN2020', 'needB'), '5.21'), (('KR2014/CN2013', 'needB'), '11.56')]},
    # Table 2
    **{('t2_age', w, ''): v for w, v in [('KR2014', '78.5'), ('CN2015', '76.0'), ('KR2018', '78.8'), ('CN2018', '76.3'),
                                          ('KR2020', '79.8'), ('CN2020', '77.2')]},
    **{('t2_female', w, ''): v for w, v in [('KR2014', '58.2'), ('CN2015', '61.6'), ('KR2018', '55.8'), ('CN2018', '59.5'),
                                             ('KR2020', '55.1'), ('CN2020', '60.1')]},
    **{('t2_75plus', w, ''): v for w, v in [('KR2014', '67.3'), ('CN2015', '52.3'), ('KR2018', '67.8'), ('CN2018', '54.2'),
                                             ('KR2020', '70.8'), ('CN2020', '58.6')]},
    # all adults aged 65+ (S15)
    **{('all_need', w, ''): v for w, v in [('KR2014', '16.6'), ('KR2018', '14.2'), ('KR2020', '14.4'), ('CN2013', '27.3'),
                                            ('CN2015', '27.7'), ('CN2018', '29.3'), ('CN2020', '23.3')]},
    **{('all_nohelp', w, ''): v for w, v in [('KR2014', '4.0'), ('KR2018', '3.9'), ('KR2020', '4.9'), ('CN2013', '3.7'),
                                              ('CN2015', '4.6'), ('CN2018', '4.4'), ('CN2020', '2.6')]},
}
PUB_BOOT = {  # (estimate, lower, upper) for bootstrap statistics
    ('gap', 'KR2014/CN2015', 'needA'): ('8.11', '3.5', '12.8'), ('gap', 'KR2018/CN2018', 'needA'): ('13.50', '8.7', '18.2'),
    ('gap', 'KR2020/CN2020', 'needA'): ('24.34', '19.7', '29.3'), ('gap', 'KR2014/CN2013', 'needA'): ('11.82', '8.0', '15.8'),
    ('crude', 'KR2014/CN2015', 'needA'): ('7.5', '3.0', '11.7'), ('crude', 'KR2018/CN2018', 'needA'): ('12.6', '8.1', '17.0'),
    ('crude', 'KR2020/CN2020', 'needA'): ('23.1', '18.7', '27.5'), ('crude', 'KR2014/CN2013', 'needA'): ('10.8', '7.0', '14.8'),
    ('gap', 'KR2014/CN2015', 'needB'): ('4.72', '-0.7', '10.2'), ('gap', 'KR2018/CN2018', 'needB'): ('9.33', '3.3', '15.5'),
    ('gap', 'KR2020/CN2020', 'needB'): ('14.75', '8.9', '21.4'), ('gap', 'KR2014/CN2013', 'needB'): ('5.63', '0.3', '11.3'),
    ('gap', 'KR2014/CN2015', 'needC'): ('6.4', '1.8', '11.2'), ('gap', 'KR2018/CN2018', 'needC'): ('9.8', '5.3', '14.9'),
    ('gap', 'KR2020/CN2020', 'needC'): ('21.8', '16.4', '26.9'), ('gap', 'KR2014/CN2013', 'needC'): ('10.1', '5.8', '14.4'),
    ('gap', 'KR2014/CN2015', 'needM'): ('4.1', '-1.0', '9.8'), ('gap', 'KR2018/CN2018', 'needM'): ('8.0', '1.9', '14.7'),
    ('gap', 'KR2020/CN2020', 'needM'): ('13.7', '7.6', '20.2'),
    ('A-B', 'KR2014/CN2015', ''): ('3.4', '-1.8', '8.7'), ('A-B', 'KR2018/CN2018', ''): ('4.2', '-1.7', '9.9'),
    ('A-B', 'KR2020/CN2020', ''): ('9.6', '4.6', '15.5'), ('A-B', 'KR2014/CN2013', ''): ('6.2', '1.5', '10.6'),
    ('A-common', 'KR2014/CN2015', ''): ('1.7', '0.0', '3.4'), ('A-common', 'KR2018/CN2018', ''): ('3.7', '2.0', '5.6'),
    ('A-common', 'KR2020/CN2020', ''): ('2.6', '0.8', '4.5'),
    ('pairing', 'CN2013 vs CN2015', 'needA'): ('3.7', '0.2', '7.3'), ('pairing', 'CN2013 vs CN2015', 'needB'): ('0.9', '-3.0', '5.1'),
    ('gap70', 'KR2014/CN2015', 'needA'): ('11.7', '7.0', '15.9'), ('gap70', 'KR2018/CN2018', 'needA'): ('14.8', '10.8', '19.2'),
    ('gap70', 'KR2020/CN2020', 'needA'): ('20.4', '15.7', '24.9'), ('gap70', 'KR2014/CN2013', 'needA'): ('14.1', '9.8', '18.5'),
    ('gap_men', 'KR2014/CN2015', 'needA'): ('2.2', '-5.2', '8.9'), ('gap_men', 'KR2018/CN2018', 'needA'): ('9.9', '3.9', '16.1'),
    ('gap_men', 'KR2020/CN2020', 'needA'): ('30.6', '23.8', '37.4'),
    ('gap_women', 'KR2014/CN2015', 'needA'): ('12.1', '6.2', '17.6'), ('gap_women', 'KR2018/CN2018', 'needA'): ('16.2', '10.3', '22.8'),
    ('gap_women', 'KR2020/CN2020', 'needA'): ('19.6', '13.3', '26.0'),
    ('all65', 'KR2014/CN2015', 'needA'): ('-1.0', '-2.0', '0.0'), ('all65', 'KR2018/CN2018', 'needA'): ('-0.7', '-1.5', '0.1'),
    ('all65', 'KR2020/CN2020', 'needA'): ('2.2', '1.4', '3.1'), ('all65', 'KR2014/CN2013', 'needA'): ('0.0', '-0.8', '0.9'),
    ('change', 'CN2013->2015', 'needA'): ('3.1', '-0.1', '6.6'), ('change', 'CN2015->2018', 'needA'): ('-1.6', '-4.7', '1.5'),
    ('change', 'CN2018->2020', 'needA'): ('-3.3', '-5.7', '-0.6'), ('change', 'KR2014->2018', 'needA'): ('3.8', '-1.1', '8.7'),
    ('change', 'KR2018->2020', 'needA'): ('7.8', '2.5', '12.9'),
    ('change', 'CN2013->2015', 'needB'): ('0.0', '-4.4', '4.2'), ('change', 'CN2015->2018', 'needB'): ('-3.0', '-6.5', '0.1'),
    ('change', 'CN2018->2020', 'needB'): ('-3.6', '-6.5', '-0.7'), ('change', 'KR2014->2018', 'needB'): ('1.1', '-5.3', '8.0'),
    ('change', 'KR2018->2020', 'needB'): ('1.5', '-5.9', '8.8'),
}

def _matches(raw, pub):
    """True if raw rounds to the published value at the published number of decimals."""
    if isinstance(pub, int):
        return int(round(raw)) == pub
    dec = len(pub.split('.')[1]) if '.' in pub else 0
    return bool(np.isclose(round(float(raw) + 0.0, dec), float(pub), atol=1e-9))

def charls_post_hoc(W):
    """Post hoc CHARLS-only analyses (Supplementary Text S12): help counted only on activities with need, number of help
    questions answered, respondents with difficulty only, telephone help, institutional residents, age at interview, and need
    thresholds on 2+ or 3+ of the 10 items among all adults aged 65+ (weighted)."""
    out = {}
    wm = lambda x, w: 100 * float((x * w).sum() / w.sum())
    for y in CHARLS_YEARS:
        k = f'CN{y}'
        if k not in W:
            continue
        d = W[k].df
        a = d[d['needA'] & d['hans_need']]
        out[('ph_nohelp_A_needrule', k)] = wm(~a['helped_need'], a['w'])
        b = d[d['needB'] & d['hans_need']]
        out[('ph_nohelp_B_needrule', k)] = wm(~b['helped_need'], b['w'])
        m = d[d['needA'] & d['hans']]
        for s, sel in (('1', m['n_lim'] == 1), ('2', m['n_lim'] == 2), ('3+', m['n_lim'] >= 3)):
            out[(f'ph_mean_questions_{s}', k)] = float(m.loc[sel, 'nq'].mean())
            g = a[(a['n_lim'] == int(s[0])) if s != '3+' else (a['n_lim'] >= 3)]
            out[(f'ph_nohelp_needrule_{s}', k)] = wm(~g['helped_need'], g['w'])
        # need threshold: respondents reporting only difficulty (rating 2, no item with need) and the share q of them that would
        # have to count as having need for the crude gap to vanish, given the published Korean crude share of the paired wave
        o = d[d['needD'] & ~d['needA'] & d['hans']]
        out[('ph_nohelp_Donly', k)] = wm(o['nh'] == 1, o['w'])
        kr = {'CN2015': 24.24, 'CN2018': 27.50}.get(k)   # Korea 2014 and 2018, definition A (Supplementary Table S8b)
        if kr is not None:
            pa, po = wm(m['nh'] == 1, m['w']) / 100, out[('ph_nohelp_Donly', k)] / 100
            out[('ph_qstar', k)] = (kr / 100 - pa) * m['w'].sum() / ((po - kr / 100) * o['w'].sum())
        # telephone help, which the main CHARLS help rule omits and the single KLoSA question covers
        out[('ph_nohelp_A_phone', k)] = wm((m['nh'] == 1) & ~m['helped_phone'], m['w'])
        # institutional residents (nursing home or hospital at interview) excluded
        out[('ph_n_inst', k)] = int(m['inst'].sum())
        out[('ph_nohelp_A_noinst', k)] = wm(m.loc[~m['inst'], 'nh'] == 1, m.loc[~m['inst'], 'w'])
        # age at interview (Harmonized RwAGEY, 2013-2018) instead of survey year minus birth year: drops analytic-sample members
        # under 65 at interview; in these files nobody aged 65+ at interview is under 65 by year minus birth year (checked)
        if m['agey'].notna().any():
            g = m[m['agey'] >= 65]
            out[('ph_nohelp_A_agey', k)] = wm(g['nh'] == 1, g['w'])
        allad = d[~(d['needA'] & ~d['hans'])]
        out[('ph_n_all', k)] = len(allad)
        for t in (2, 3):
            nk = allad['n_lim'] >= t
            out[(f'ph_prev_ge{t}', k)] = wm(nk, allad['w'])
            out[(f'ph_pop_ge{t}', k)] = wm(nk & ~allad['helped'], allad['w'])
            out[(f'ph_pop_ge{t}_needrule', k)] = wm(nk & ~allad['helped_need'], allad['w'])   # help counted only on activities with need
    return out

# Published post hoc CHARLS values (Supplementary Text S12); all depend on CHARLS alone
PUB_POSTHOC = {
    **{('ph_nohelp_A_needrule', k): v for k, v in [('CN2013', '14.8'), ('CN2015', '18.7'), ('CN2018', '16.6'), ('CN2020', '12.4')]},
    **{('ph_nohelp_B_needrule', k): v for k, v in [('CN2013', '13.8'), ('CN2015', '15.6'), ('CN2018', '11.4'), ('CN2020', '6.4')]},
    ('ph_mean_questions_1', 'CN2015'): '2.3', ('ph_mean_questions_2', 'CN2015'): '3.6', ('ph_mean_questions_3+', 'CN2015'): '6.5',
    ('ph_nohelp_needrule_1', 'CN2015'): '38.8', ('ph_nohelp_needrule_2', 'CN2015'): '11.9', ('ph_nohelp_needrule_3+', 'CN2015'): '3.0',
    ('ph_nohelp_needrule_1', 'CN2018'): '34.6', ('ph_nohelp_Donly', 'CN2015'): '75.4', ('ph_nohelp_Donly', 'CN2018'): '75.4',
    ('ph_qstar', 'CN2015'): '0.27', ('ph_qstar', 'CN2018'): '0.61',
    **{('ph_n_all', k): v for k, v in [('CN2013', 5326), ('CN2015', 6365), ('CN2018', 7212), ('CN2020', 8027)]},
    **{('ph_prev_ge2', k): v for k, v in [('CN2013', '17.5'), ('CN2015', '16.7'), ('CN2018', '18.5'), ('CN2020', '15.6')]},
    **{('ph_prev_ge3', k): v for k, v in [('CN2013', '12.4'), ('CN2015', '11.9'), ('CN2018', '13.2'), ('CN2020', '11.9')]},
    **{('ph_pop_ge2', k): v for k, v in [('CN2013', '0.8'), ('CN2015', '0.8'), ('CN2018', '1.1'), ('CN2020', '0.7')]},
    **{('ph_pop_ge3', k): v for k, v in [('CN2013', '0.3'), ('CN2015', '0.3'), ('CN2018', '0.3'), ('CN2020', '0.3')]},
    **{('ph_pop_ge2_needrule', k): v for k, v in [('CN2015', '0.9'), ('CN2018', '1.2')]},
    **{('ph_pop_ge3_needrule', k): v for k, v in [('CN2015', '0.4'), ('CN2018', '0.3')]},
    **{('ph_nohelp_A_phone', k): v for k, v in [('CN2015', '14.6'), ('CN2018', '13.4')]},
    **{('ph_n_inst', k): v for k, v in [('CN2015', 21), ('CN2018', 29)]},
    **{('ph_nohelp_A_noinst', k): v for k, v in [('CN2015', '17.1'), ('CN2018', '14.9')]},
    **{('ph_nohelp_A_agey', k): v for k, v in [('CN2015', '16.7'), ('CN2018', '14.8')]},
}

def compare(summary, est, t2=None, t_all=None, ph=None):
    """Side-by-side table of reproduced and published values. Point estimates should match at the published
    precision; bootstrap limits use a different random-number stream and can differ by a few tenths."""
    rows = []
    t2_rows = {'t2_age': 'Mean age, years', 't2_female': 'Female, %', 't2_75plus': 'Aged 75 or older, %'}
    for key, pub in PUB_POINTS.items():
        kind, item, need = key
        raw = None
        if kind in ('pct', 'se', 'n', 'n70', 'nohelp'):
            e = est.get(('prop', item, need))
            if e is not None:
                raw = {'pct': e['pct'], 'se': e['se'], 'n': e['n'], 'n70': e['n70'], 'nohelp': e['no_helper']}[kind]
        elif kind in ('kr_std', 'cn_std'):
            e = est.get(('gap', item, need))
            raw = e[kind] if e else None
        elif kind in t2_rows and t2 is not None and item in t2.columns:
            raw = t2.loc[t2_rows[kind], item]
        elif kind in ('all_need', 'all_nohelp') and t_all is not None and item in t_all.index:
            raw = t_all.loc[item, 'Need (definition A), weighted %' if kind == 'all_need' else 'Need and no helper, weighted %']
        if raw is None:
            continue
        rows.append({'statistic': kind, 'item': item, 'definition': need, 'published': pub, 'reproduced': raw,
                     'match': _matches(raw, pub), 'published_CI': '', 'reproduced_CI': ''})
    s = summary.set_index(['statistic', 'pair_or_change', 'definition'])
    for key, (pe, lo, hi) in PUB_BOOT.items():
        if key not in s.index:
            continue
        r = s.loc[key]
        rows.append({'statistic': key[0], 'item': key[1], 'definition': key[2], 'published': pe, 'reproduced': r.estimate,
                     'match': _matches(r.estimate, pe), 'published_CI': f'{lo} to {hi}' if lo else '',
                     'reproduced_CI': f'{r.lo:.1f} to {r.hi:.1f}'})
    for key, pub in PUB_POSTHOC.items():
        if ph is None or key not in ph:
            continue
        rows.append({'statistic': key[0], 'item': key[1], 'definition': '', 'published': pub, 'reproduced': ph[key],
                     'match': _matches(ph[key], pub), 'published_CI': '', 'reproduced_CI': ''})
    out = pd.DataFrame(rows)
    out['reproduced'] = out['reproduced'].map(lambda v: round(float(v), 3))
    return out

## 3. Load the data

In [ ]:
files = index_files(DATA_DIR, require_klosa=not CHARLS_ONLY)
for k, v in files.items():
    print(f'{k:12s}', v[0] if v else '(not found, optional)')
H = load_harmonized(files)
W = {}
for y in CHARLS_YEARS:
    W[f'CN{y}'] = Wave(load_charls(files, H, y), by_cluster=True)
for y in KLOSA_WAVES:
    if files.get(f'KR{y}') and not CHARLS_ONLY:
        W[f'KR{y}'] = Wave(load_klosa(files, y), by_cluster=False)
print({k: len(v.df) for k, v in W.items()}, '(respondents aged 65+ with valid items and a positive weight)')

## 4. Weighted percentage with no helper (Table 3, Supplementary Tables S1 and S8b)

In [ ]:
import pandas as pd
from IPython.display import display
est = point_estimates(W, wmean_ci, gap, crude_gap)
t1 = pd.DataFrame([{'wave': k[1], 'definition': DEF_LABEL[k[2]], **v} for k, v in est.items() if k[0] == 'prop'])
t1 = t1.sort_values(['wave', 'definition']).round(2)
t1.to_csv(f'{OUT_DIR}/table_S1_no_helper_by_definition.csv', index=False)
t1

## 5. Characteristics of respondents with need (Table 2) and need among all adults aged 65+ (Table 5, Supplementary Table S15)

In [ ]:
t2 = characteristics(W).round(1)
t2.to_csv(f'{OUT_DIR}/table2_characteristics.csv')
from IPython.display import display
display(t2)
t_all = all_adults_shares(W).round(2)
t_all.to_csv(f'{OUT_DIR}/table_S15_all_adults_65plus.csv')
t_all

## 6. Bootstrap (about 30 seconds for 1,000 replicates)

In [ ]:
reps = bootstrap(W, gap, crude_gap, B=B, seed=SEED, progress=lambda b: print(b, 'replicates'))
summary = summarize(est, reps, W, gap)
summary.round(2).to_csv(f'{OUT_DIR}/bootstrap_summary.csv', index=False)

## 7. Country gaps (Table 4, Figure 3a) and sensitivity estimates

In [ ]:
rows = []
for lab, (ky, cy, role) in PAIRS.items():
    if ('gap', lab, 'needA') not in est:
        continue
    e = est[('gap', lab, 'needA')]
    s = summary.set_index(['statistic', 'pair_or_change', 'definition'])
    g = s.loc[('gap', lab, 'needA')]; c = s.loc[('crude', lab, 'needA')]
    rows.append({'pair': lab, 'role': role, 'Korea std %': round(e['kr_std'], 1), 'China std %': round(e['cn_std'], 1),
                 'crude gap': f"{c.estimate:.1f} ({c.lo:.1f} to {c.hi:.1f})",
                 'standardized gap': f"{g.estimate:.1f} ({g.lo:.1f} to {g.hi:.1f})"})
t4 = pd.DataFrame(rows, columns=['pair', 'role', 'Korea std %', 'China std %', 'crude gap', 'standardized gap'])
t4.to_csv(f'{OUT_DIR}/table4_gaps.csv', index=False)
t4

In [ ]:
fmt = summary.assign(result=summary.apply(lambda r: f"{r.estimate:.1f} ({r.lo:.1f} to {r.hi:.1f})", axis=1))
wide = fmt.pivot_table(index=['statistic', 'definition'], columns='pair_or_change', values='result', aggfunc='first')
wide.to_csv(f'{OUT_DIR}/sensitivity_wide.csv')
wide

## 8. Additional analyses (not in the published tables)

* **Severity on a common scale.** The published severity strata count limitations on each survey's own items (17 in Korea, 10 in China). Here severity is counted on items shared by both surveys: need on 3 or more of the 10 common activities (`needC3`), and need on 2 or more of the 5 matched ADL items (`needM2`).
* **Population denominator under other definitions.** The share of all adults aged 65+ with need and no helper, and its standardized Korea-minus-China difference, under definition B, the common-item and the matched-ADL definitions (`all65`). The standard population for all `all65` estimates is the mean of the two countries' age-sex distributions among all respondents aged 65+.

In [ ]:
added = summary[summary.statistic.isin(['gap', 'all65']) & summary.definition.isin(['needC3', 'needM2', 'needB', 'needC', 'needM'])]
added = added[~((added.statistic == 'gap') & added.definition.isin(['needB', 'needC', 'needM']))].copy()
added['result'] = [f"{r.estimate:.1f} ({r.lo:.1f} to {r.hi:.1f})" for r in added.itertuples()]
props = pd.DataFrame([{'wave': k[1], 'definition': DEF_LABEL[k[2]], 'n': v['n'], 'weighted % no helper': round(v['pct'], 1),
                       '95% CI': f"{v['lo']:.1f} to {v['hi']:.1f}"} for k, v in est.items() if k[0] == 'prop' and k[2] in ('needC3', 'needM2')])
props.to_csv(f'{OUT_DIR}/added_severity_common_scale_proportions.csv', index=False)
added.to_csv(f'{OUT_DIR}/added_gaps_severity_and_population.csv', index=False)
pop = all_adults_shares(W, needs=('needA', 'needB', 'needC', 'needM')).round(2)
pop.to_csv(f'{OUT_DIR}/added_population_shares_by_definition.csv', index=False)
display(props); display(added[['statistic', 'pair_or_change', 'definition', 'result']]); pop

## 9. Check against the published values

In [ ]:
ph = charls_post_hoc(W)
pd.Series({f'{k[0]} {k[1]}': round(v, 2) for k, v in ph.items()}).to_csv(f'{OUT_DIR}/posthoc_charls.csv')
cmp = compare(summary, est, t2=characteristics(W), t_all=all_adults_shares(W), ph=ph)
cmp.to_csv(f'{OUT_DIR}/comparison_with_published.csv', index=False)
print(f'{int(cmp.match.sum())} of the {len(cmp)} published values checked are reproduced at the published precision')
print('Not matching:'); print(cmp.loc[~cmp.match, ['statistic', 'item', 'definition', 'published', 'reproduced']].to_string())
cmp

### KLoSA checks for the authors (outside the automated comparison)

With the KLoSA files, this table reports for every Korean wave the weighted and unweighted no-helper shares (to see how much of a change between waves comes from the weights), C218_01 = 57 among respondents with need (listed in the 2018 and 2020 codebooks but not offered by the questionnaire; the main rule counts only 56 as no helper), 2014 refresh-panel members aged 65 or older (expected: none), LTCI use (C320n = 1) among respondents with need (published: 4.3%, 7.9% and 14.0%), users of institutional LTCI benefits (C327n = 1) and the share with no helper without them, proxy interviews for the ADL and helper module (C225 = 4; published shares without proxies: 26.0%, 29.0% and 36.4%), and design effects of the no-helper share when SEs are clustered by the enumeration district (ed) listed in the 2014-2022 codebooks (the published Korean SEs treat respondents as independent).

In [ ]:
if any(k.startswith('KR') for k in W):
    diag = klosa_diagnostics(W, wmean_ci)
    diag.to_csv(f'{OUT_DIR}/klosa_diagnostics.csv')
    print(diag.round(2).to_string())

## 10. Figures

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, ctry, defs in ((axes[0], 'KR', ['needA', 'needB']), (axes[1], 'CN', ['needA', 'needB', 'needD'])):
    for j, nd in enumerate(defs):
        pts = sorted((int(k[1][2:]), v) for k, v in est.items() if k[0] == 'prop' and k[1].startswith(ctry) and k[2] == nd)
        if not pts:
            continue
        x = np.array([p[0] for p in pts]) + (j - 1) * 0.15
        y = np.array([p[1]['pct'] for p in pts])
        err = np.array([[p[1]['pct'] - p[1]['lo'] for p in pts], [p[1]['hi'] - p[1]['pct'] for p in pts]])
        ax.errorbar(x, y, yerr=err, fmt='o', capsize=3, label=DEF_LABEL[nd])
    ax.axvspan(2019.5, 2020.5, color='0.9', zorder=0)
    ax.set_title('(a) Korea (KLoSA)' if ctry == 'KR' else '(b) China (CHARLS)')
    ax.set_xlabel('Survey year'); ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel('Weighted % with no helper (95% CI)')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/figure2_no_helper_by_definition.png', dpi=300)

s = summary.set_index(['statistic', 'pair_or_change', 'definition'])
labs = [l for l in PAIRS if ('gap', l, 'needA') in s.index]
fig, ax = plt.subplots(figsize=(7, 4))
for j, kind in enumerate(['crude', 'gap']):
    vals = [s.loc[(kind, l, 'needA')] for l in labs]
    x = np.arange(len(labs)) + (j - 0.5) * 0.25
    ax.errorbar(x, [v.estimate for v in vals], yerr=[[v.estimate - v.lo for v in vals], [v.hi - v.estimate for v in vals]],
                fmt='o' if kind == 'crude' else 's', capsize=3, label='Crude' if kind == 'crude' else 'Age-sex standardized')
ax.axhline(0, color='0.5', lw=0.8)
ax.set_xticks(range(len(labs))); ax.set_xticklabels([l.replace('/', ' / ') for l in labs], fontsize=8)
ax.set_ylabel('Korea minus China, percentage points'); ax.legend(frameon=False)
fig.tight_layout()
if labs:
    fig.savefig(f'{OUT_DIR}/figure3a_gap.png', dpi=300)
plt.show()

## 11. Software versions

In [ ]:
import platform, numpy, pandas, matplotlib
print('Python', platform.python_version(), '| pandas', pandas.__version__, '| numpy', numpy.__version__, '| matplotlib', matplotlib.__version__)
print('Outputs written to', OUT_DIR, ':', sorted(os.listdir(OUT_DIR)))